<a href="https://colab.research.google.com/github/RobJavVar/DataSciencePsychNeuro/blob/master/ExerciseSubmissions/15_power-analysis-via-simulations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 15: Power analyses

This  assignment is designed to give you practice with Monte Carlo methods to conduct power analyses via simulation. You won't need to load in any data for this homework. We will, however, be using parts of the homework from last week.

---
## 1. Simulating data (1 points)


Pull your `simulate_data()` function from your last homework and add it below.

As a reminder, this function simulates the relationship between age, word reading experience, and reading comprehension skill.

`c` is reading comprehension, and `x` is word reading experience.

In [3]:
sample_size = 100 # How many children in data set?
age_lo = 80     # minimum age, in months
age_hi = 200    # maximum age, in months
beta_xa = 0.5   # amount by which experience changes for increase of one month in age
beta_x0 = -5    # amount of experience when age = 0 (not interpretable, since minimum age for this data is 80 months)
sd_x = 50       # standard dev of gaussian noise term, epsilon_x
beta_ca = 0.8   # amount that comprehension score improves for every increase of one unit in age
beta_cx = 3     # amount that comprehension score improves for every increase of one unit in reading experience
beta_c0 = 10    # comprehension score when reading experience is 0.
sd_c = 85      # standard dev of gaussian noise term, epsilon_c

simulate_data <- function(sample_size, age_lo, age_hi, beta_xa, beta_x0, sd_x, beta_ca, beta_cx, beta_c0, sd_c) {
age <- runif(sample_size, min = age_lo, max = age_hi) # generate ages from 80-200 months
epsilon_x <- rnorm(sample_size, mean = 0 , sd = sd_x) # generate reading experience with gaussian noise term added as older aged kids have more experience reading thus better comprehension
x <- beta_xa * age  + beta_x0 + epsilon_x
epsilon_c <- rnorm(sample_size, mean = 0, sd = sd_c)
c <- beta_ca * age + beta_cx * x + beta_c0 + epsilon_c

      return(data.frame(age=age,x=x,c=c)) # it's actually bad form to have a variable named "c" in R, my bad...
}

dat <- simulate_data(sample_size, age_lo, age_hi, beta_xa, beta_x0, sd_x, beta_ca, beta_cx, beta_c0, sd_c)
head(dat)

,age,x,c
,<dbl>,<dbl>,<dbl>
1,83.63734,10.10594,70.27101
2,189.98455,10.04969,344.64459
3,101.26692,80.69134,215.31102
4,93.73569,37.95668,285.21027
5,93.71419,58.97700,141.34092
6,184.86653,80.27251,443.75728


---
## 2. `run_analysis()` function (2 pts)

Last week, we looked at whether word reading experience(`x`) mediated the relation between `age` and reading comprehension (`c`).

Now we're going to use our `simulate_data()` function to conduct a power analysis. The goal is to determine how many participants we would need in order to detect both the mediated and the direct effects in this data.

*Note: We're going to pretend for the sake of simplicity that we don't have any control over the ages of the children we get (so ages are generated using `runif(sample_size, age_lo, age_hi)`, although of course this would be an unusual situation in reality.*

First, write a function, `run_analysis()`, that takes in simulated data, runs **your mediation from last week**, and returns a vector containing the ACME and ADE estimates and p-values (these are the `d0`, `d0.p`, `z0`, and `z0.p` features of the mediated model object, e.g., `fitMed$d0.p`). Print this function's output for the data we simulated previously.

In [6]:
library(mediation)
library(MASS)


### Mediation from exercise 14
fitM <- lm(x ~ age, data = dat)
fitY <- lm(c ~ age + x, data = dat)


summary(fitM)
summary(fitY)

fitMed <- mediate(fitM, fitY, treat = "age", mediator = "x")
summary(fitMed)

In [9]:
run_analysis <- function(dat) {
fitM <- lm(x ~ age, data = dat)
fitY <- lm(c ~ age + x, data = dat)
    
#summary(fitM)
#summary(fitY)
    
fitMed <- mediate(fitM, fitY, treat = "age", mediator = "x", sims = 500)
summary(fitMed)
d0 <- fitMed$d0 #acme
d0.p <- fitMed$d0.p #acme p val
z0 <- fitMed$z0 #ade 
z0.p <- fitMed$z0.p #ade p val

return(c(ACME = d0, ACME_p = d0.p, ADE = z0, ADE_p = z0.p))
    }


---
## 3. `repeat_analysis()` function (3 pts)

Next fill in the function `repeat_analysis()` below so that it simulates and analyzes data `num_simulations` times. Store the outputs from each simulation in the `simouts` matrix. Calculate and return the coverage across all the simulations run for both ACME and ADE.

In [11]:
repeat_analysis <- function(num_simulations, alpha, sample_size, age_lo, age_hi,
        beta_xa, beta_x0, sd_x, beta_ca, beta_cx, beta_c0, sd_c) {
    # Initialize simouts matrix for storing each output from run_analysis()
    #d0.sims <- fitMed$d0.sims #acme sims
    #z0.sims <- fitMed$z0.sims #ade sims
    simouts <- matrix(rep(NA, num_simulations*4), nrow=num_simulations, ncol=4)

    
    # Start simulating
    for (i in 1:num_simulations) {

        dat <- simulate_data(sample_size, age_lo, age_hi, beta_xa, beta_x0, sd_x, beta_ca, beta_cx, beta_c0, sd_c)
        simouts[i, ] <- run_analysis(dat) 

    }

    # Calculate coverage for both ACME and ADE estimates using p-values in simouts
    ACME_cov = mean(simouts[,2] <= alpha)
    ADE_cov =  mean(simouts[,4] <= alpha)
   

    return(list(ACME_cov = ACME_cov, ADE_cov = ADE_cov))
}





Now run the `repeat_analysis()` function using the same parameter settings as above, for 10 simulations, with an alpha criterion of 0.01.

In [18]:
results <- repeat_analysis(
sample_size = sample_size,
  age_lo = age_lo,
  age_hi = age_hi,
  beta_xa = beta_xa,
  beta_x0 = beta_x0,
  sd_x = sd_x,
  beta_ca = beta_ca,
  beta_cx = beta_cx,
  beta_c0 = beta_c0,
  sd_c = sd_c,  
num_simulations = 10, #10 simulations fixed
alpha = 0.01 )   #0.01 alpha value 
   

print(results)

$ACME_cov
[1] 0.8

$ADE_cov
[1] 0.7



---
## 4. Testing different sample sizes (2 pts)

Finally, do the same thing (10 simulations, alpha criterion of 0.01) but for 5 different sample sizes: 50, 75, 100, 125, 150. You can do this using `map` (as in the tutorial), or a simple `for` loop, or by calculating each individually. Up to you! This should take around 3 minutes to run.

In [23]:
sample_sizes <- c(50, 75, 100, 125, 150) #list of sample size
power_results <- data.frame(sample_size = sample_sizes, 
                             ACME_cov   = NA, 
                             ADE_cov    = NA)

for (i in 1:length(sample_sizes)) {
  res <- repeat_analysis(
    num_simulations = 10,
    alpha = 0.01,
sample_size = sample_sizes[i], #iterate through list
  age_lo = age_lo,
  age_hi = age_hi,
  beta_xa = beta_xa,
  beta_x0 = beta_x0,
  sd_x = sd_x,
  beta_ca = beta_ca,
  beta_cx = beta_cx,
  beta_c0 = beta_c0,
  sd_c = sd_c 
  )
  power_results$ACME_cov[i] <- res$ACME_cov
  power_results$ADE_cov[i]  <- res$ADE_cov
}

Print your results.

In [24]:
print(power_results)

  sample_size ACME_cov ADE_cov
1          50      0.6     0.4
2          75      0.9     0.4
3         100      0.7     0.5
4         125      0.9     0.7
5         150      0.9     0.9


## 5. Reflection (2 pts)

If this were a real power analysis, we'd want to run more simulations per sample size (to get a more precise estimate of power) and we may also want to test out some other values of the parameters we used to simulate our data. However, what would you conclude just based on the results above?

>  The ACME, which assesses the indirect mediation effect, reached 0.9 power at n = 75 and maintained that power through n = 150 sample size, suggesting the mediated effect was detected even with a smaller sample size. The ADE, which assesses the direct effect, showed a consistent pattern of increasing power with increased sample size — with the power at 0.4 at n = 50 to 0.9 at n = 150. This shows that a larger sample size is needed to  detect the direct effect, and so n = 150 sample size would be the needed sample size to achieve a power of 0.9 for both effects. 


**Given** how we generated the data, why was the direct effect harder to detect than the mediated effect?
> The direct effect was more difficult to detect than the mediated effect because of the difference in coefficients we had initialized set by the beta-weights. Having a difference in coefficients changed the effect strength so a larger coefficient for ACME meant having a stronger effect thus making it more easily detectable to detect than a weaker effect. For ADE, in initializing a lower coefficient to ADE we made it increasingly more difficult to be detected. 

**DUE:** 5pm EST, March 31, 2026

**IMPORTANT** Did you collaborate with anyone on this assignment? If so, list their names here.
> *Someone's Name*